In [ ]:
pip install fastapi uvicorn redis numpy httpx

In [10]:
import subprocess
import time

print("Installing Redis...")
install_process = subprocess.run(['sudo', 'apt', 'install', '-y', 'redis-server'], capture_output=True, text=True)
print(install_process.stdout)
if install_process.stderr:
    print("Stderr during install:\n", install_process.stderr)
print("Redis installation complete.")

print("Starting Redis server...")
# Start Redis server as a daemon in the background
# Using subprocess.Popen to avoid blocking the notebook cell
try:
    start_process = subprocess.Popen(['redis-server', '--daemonize', 'yes'], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    stdout, stderr = start_process.communicate(timeout=5)
    print(stdout)
    if stderr:
        print("Stderr during Redis server start:\n", stderr)
    print("Redis server command initiated.")
except Exception as e:
    print(f"Error starting Redis server: {e}")

# Give Redis a moment to start up
time.sleep(2)

# Verify Redis is running
print("Verifying Redis server status...")
try:
    ping_process = subprocess.run(['redis-cli', 'ping'], capture_output=True, text=True, check=True)
    if ping_process.stdout.strip() == 'PONG':
        print("Redis server is running successfully!")
    else:
        print(f"Redis-cli ping response: {ping_process.stdout.strip()}")
        print("Redis server might not be running correctly.")
except subprocess.CalledProcessError as e:
    print(f"Error checking Redis status: {e}")
    print(f"Stderr: {e.stderr}")
    print(f"Stdout: {e.stdout}")
except Exception as e:
    print(f"An unexpected error occurred during Redis status check: {e}")

Installing Redis...
Reading package lists...
Building dependency tree...
Reading state information...
redis-server is already the newest version (5:6.0.16-1ubuntu1.1).
0 upgraded, 0 newly installed, 0 to remove and 1 not upgraded.

Stderr during install:
 


Redis installation complete.
Starting Redis server...

Redis server command initiated.
Verifying Redis server status...
Redis server is running successfully!


In [2]:
import redis
import json
import numpy as np
import time

r = redis.Redis(host='localhost', port=6379, db=0, decode_responses=True)

CACHE_TTL = 3600  # 1 hour

In [14]:
def get_embedding(text):
    cache_key = f"embedding:{text}"

    cached = r.get(cache_key)
    if cached:
        return np.array(json.loads(cached))

    # 🔁 Replace with OpenAI embedding call
    embedding = fake_embedding(text)

    # Use r.set() with 'ex' argument instead of deprecated r.setex()
    r.set(cache_key, json.dumps(embedding.tolist()), ex=CACHE_TTL)
    return embedding


def fake_embedding(text):
    # dummy vector (replace with real API)
    return np.random.rand(768)

In [15]:
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))


def check_semantic_cache(query_embedding):
    keys = r.keys("response:*")

    for key in keys:
        data = json.loads(r.get(key))

        cached_embedding = np.array(data["embedding"])
        similarity = cosine_similarity(query_embedding, cached_embedding)

        if similarity > 0.95:
            return data["response"]

    return None


def store_response(query, embedding, response):
    key = f"response:{query}"
    data = {
        "embedding": embedding.tolist(),
        "response": response
    }
    # Use r.set() with 'ex' argument instead of deprecated r.setex()
    r.set(key, json.dumps(data), ex=CACHE_TTL)

In [5]:
from fastapi import FastAPI, BackgroundTasks
import uuid
import asyncio

app = FastAPI()

job_store = {}

In [6]:
@app.post("/ask")
async def ask(query: str):
    embedding = get_embedding(query)

    # 🔍 Semantic Cache Check
    cached_response = check_semantic_cache(embedding)
    if cached_response:
        return {"source": "cache", "answer": cached_response}

    # 🤖 Simulate LLM call
    response = await call_llm(query)

    store_response(query, embedding, response)

    return {"source": "llm", "answer": response}


async def call_llm(query):
    await asyncio.sleep(2)  # simulate delay
    return f"Answer for: {query}"

In [7]:
@app.post("/research")
async def research(query: str, background_tasks: BackgroundTasks):
    job_id = str(uuid.uuid4())

    job_store[job_id] = {"status": "processing", "result": None}

    background_tasks.add_task(run_research, job_id, query)

    return {"job_id": job_id}


async def run_research(job_id, query):
    await asyncio.sleep(5)  # simulate long task

    result = f"Deep research result for: {query}"

    job_store[job_id] = {
        "status": "completed",
        "result": result
    }


@app.get("/research/status/{job_id}")
async def get_status(job_id: str):
    return job_store.get(job_id, {"status": "not_found"})

In [12]:
import threading
import requests
import time

URL = "http://127.0.0.1:8000/ask"

def send_request():
    res = requests.post(URL, params={"query": "What is AI?"})
    print(res.json())

threads = []
start = time.time()

for _ in range(10):
    t = threading.Thread(target=send_request)
    threads.append(t)
    t.start()

for t in threads:
    t.join()

end = time.time()

print(f"Total time: {end - start:.2f}s")

/tmp/ipykernel_2563/402821590.py:11: DeprecationWarning: Call to deprecated setex. (Use 'set' instead.) -- Deprecated since version 2.6.12.
  r.setex(cache_key, CACHE_TTL, json.dumps(embedding.tolist()))


INFO:     127.0.0.1:37052 - "POST /ask?query=What+is+AI%3F HTTP/1.1" 200 OK
{'source': 'llm', 'answer': 'Answer for: What is AI?'}
INFO:     127.0.0.1:37068 - "POST /ask?query=What+is+AI%3F HTTP/1.1" 200 OK
INFO:     127.0.0.1:37070 - "POST /ask?query=What+is+AI%3F HTTP/1.1" 200 OK
INFO:     127.0.0.1:37072 - "POST /ask?query=What+is+AI%3F HTTP/1.1" 200 OK
{'source': 'llm', 'answer': 'Answer for: What is AI?'}
{'source': 'llm', 'answer': 'Answer for: What is AI?'}
{'source': 'llm', 'answer': 'Answer for: What is AI?'}
INFO:     127.0.0.1:37078 - "POST /ask?query=What+is+AI%3F HTTP/1.1" 200 OK
INFO:     127.0.0.1:37090 - "POST /ask?query=What+is+AI%3F HTTP/1.1" 200 OK
INFO:     127.0.0.1:37098 - "POST /ask?query=What+is+AI%3F HTTP/1.1" 200 OK
INFO:     127.0.0.1:37106 - "POST /ask?query=What+is+AI%3F HTTP/1.1" 200 OK
INFO:     127.0.0.1:37116 - "POST /ask?query=What+is+AI%3F HTTP/1.1" 200 OK
INFO:     127.0.0.1:37122 - "POST /ask?query=What+is+AI%3F HTTP/1.1" 200 OK
{'source': 'llm', 'a

/tmp/ipykernel_2563/1357654673.py:26: DeprecationWarning: Call to deprecated setex. (Use 'set' instead.) -- Deprecated since version 2.6.12.
  r.setex(key, CACHE_TTL, json.dumps(data))


In [13]:
import time

def measure(query):
    start = time.time()
    response = requests.post(URL, params={"query": query})
    end = time.time()
    return end - start


q = "Explain machine learning"

no_cache = measure(q)
embed_cache = measure(q)
full_cache = measure(q)

print(f"""
No Cache: {no_cache:.2f}s
Embedding Cache: {embed_cache:.2f}s
Full Cache: {full_cache:.2f}s
""")

/tmp/ipykernel_2563/402821590.py:11: DeprecationWarning: Call to deprecated setex. (Use 'set' instead.) -- Deprecated since version 2.6.12.
  r.setex(cache_key, CACHE_TTL, json.dumps(embedding.tolist()))


INFO:     127.0.0.1:57000 - "POST /ask?query=Explain+machine+learning HTTP/1.1" 200 OK
INFO:     127.0.0.1:57002 - "POST /ask?query=Explain+machine+learning HTTP/1.1" 200 OK
INFO:     127.0.0.1:57004 - "POST /ask?query=Explain+machine+learning HTTP/1.1" 200 OK

No Cache: 2.01s
Embedding Cache: 0.01s
Full Cache: 0.01s



/tmp/ipykernel_2563/1357654673.py:26: DeprecationWarning: Call to deprecated setex. (Use 'set' instead.) -- Deprecated since version 2.6.12.
  r.setex(key, CACHE_TTL, json.dumps(data))


In [11]:
import uvicorn
import threading
import time

def run_uvicorn():
    uvicorn.run(app, host="0.0.0.0", port=8000)

# Start Uvicorn in a separate thread
uvicorn_thread = threading.Thread(target=run_uvicorn, daemon=True)
uvicorn_thread.start()

print("Uvicorn server started in background.")
# Give the server a moment to start up
time.sleep(3)

INFO:     Started server process [2563]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


Uvicorn server started in background.
